In [18]:
import pandas as pd

# lire les données depuis le fichier
def lire_donnees(fichier, colonnes_X, colonne_Y):
    data = pd.read_csv(fichier)
    
    # extraction des colonnes qu'on va utiliser comme features et la target
    # depuis notre fichier, sous forme de listes pour travailler facilement après
    features = data[colonnes_X].values.tolist()
    target = data[colonne_Y].values.tolist()
    
    # features :liste de listes contenant les features et chaque sous-liste correspond à une ligne du dataset
    # ajouter le biais (1) au début de chaque ligne c'est à dire une colonne des 1 juste pour que w0 soit pris en compte dans le calcul
    for ligne in features: 
        ligne.insert(0, 1)
    
    return features, target 


# initialiser les poids à 0 et nb_colonnes : c'est le nombre de valeurs dans une ligne de X (biais + features)
def init_poids(nb_colonnes):
    return [0.0 for _ in range(nb_colonnes)]


# calcul de la prédiction pour une ligne
def predire(x, w):
    s = 0
    for i in range(len(w)):
        s += x[i] * w[i]
    return s


# calcul de l'erreur (MSE): l'argument Y représente les valeurs réelles du target de notre dataset
# w : poids du modèle
def erreur(X, Y, w):
    m = len(Y) # nombre de lignes dans dataset
    somme = 0
    # pour chaque ligne :
    # on calcule la prédiction y_pred
    # on calcule l'erreur (y_pred - y réel)
    # on élève au carré pour éviter les valeurs négatives et on fait la somme de tous les erreurs au carré
    for i in range(m):
         #on sait que la fonction prédire calcule la prédiction pour une seule ligne alors on fait entrer une seule ligne X[i] à chaque itération
        y_pred = predire(X[i], w)
        somme += (y_pred - Y[i])**2
    
    # on divise par 2m pour avoir la formule de MSE , ce 2m facilite le calcul après quand on passe au gradient
    return somme / (2*m)


# mise à jour des poids (gradient descent)
def mise_a_jour(X, Y, w, alpha):
    m = len(Y)  # nombre d'exemples (lignes du dataset)
    n = len(w)  # nombre de poids (biais + features)

    # liste ou on va stocker le gradient de chaque poids
    # au début on met des 0 car on va les calculer après
    grads = [0.0] * n
    
    # calcul des gradients
    for j in range(n): #on parcourt chaque poids w[j], on fixe le poids
        s = 0
        for i in range(m): #on parcourt toutes les lignes du dataset
            # calcul de l'erreur pour la ligne i (différence entre prédiction et valeur réelle)
            s += (predire(X[i], w) - Y[i]) * X[i][j] # X[i][j] représente la valeur de la feature j pour la ligne i
            
        # on fait la moyenne sur toutes les lignes: c"est la formule de gradient pour un poids précis  w[j]
        grads[j] = s / m 
    
    # mise à jour: on corrige chaque poids dans la direction opposée du gradient
    for j in range(n):
        w[j] = w[j] - alpha * grads[j]
    # cette fonction retourne la liste des poids après mise à jour
    return w

# fonction d'entraînement
# elle permet d'apprendre les meilleurs poids w
# en réduisant progressivement l'erreur grâce à la descente de gradient
def entrainer(X, Y, alpha=0.1, nb_iter=1000):
    
    #nombre de colonnes dans X (biais + features)
    nb_features = len(X[0])

    # on initialise les poids à 0 au début de l'apprentissage avec la fonction qu'on a déja défini
    w = init_poids(nb_features)
    
    # boucle principale
    for it in range(nb_iter):
        
        # on met à jour les poids avec l'appel de la fonction de mise à jour des poids (descente de gradient)
        w = mise_a_jour(X, Y, w, alpha)
        
        # afficher de temps en temps pour voir si ça diminue
        if it % 100 == 0:
            
            # on calcule l'erreur actuelle du modèle
            e = erreur(X, Y, w)
            # ce print est juste pour mieux visualiser si l'erreur diminue (donc si le modèle apprend)
            print("itération", it, "| erreur =", round(e, 4))
            
    # on retourne les poids finaux appris par le modèle
    return w



# J'ai décidé de créer une fonction globale d'entraînement du modèle qui fait appel aux fonctions déjà définies en dessous
# elle permet de charger les données, entraîner le modèle et retourner les poids appris

# J’ai choisi ces 3 arguments pour rendre la fonction flexible et simple à utiliser,
# en permettant de définir les features et la target selon le besoin.
def train_model(file_path, features, target):
    
    # charger les données à partir du fichier
    X, Y = lire_donnees(file_path, features, target)
    
    # entraînement du modèle avec descente de gradient
    # alpha = vitesse d'apprentissage
    # nb_iter = nombre d'itérations pour améliorer les poids
    w = entrainer(X, Y, alpha=0.0000001, nb_iter=3000)
    # J’ai testé plusieurs valeurs du taux d’apprentissage (alpha).
    # J’ai observé que :
    # - des valeurs trop grandes provoquent une divergence (overflow / NaN)
    # - des valeurs plus petites stabilisent l’apprentissage
    # - mais ralentissent la convergence
    # J’ai donc choisi alpha = 0.0000001 avec 3000 itérations,
    # l’erreur diminue progressivement, mais très lentement après un certain nombre d’itérations
    
    # retour des poids appris par le modèle
    return w
if __name__ == "__main__":
    features = [
        'Year',
        'Dietary_Energy_Adequacy',
        'Prevalence_of_Undernourishment_in_percent',
        'Consumer_Price_Index_in_percent'
    ]
    
    target = "Number_of_Undernourished_People_in_percent"
    
    w = train_model("../data/data_nutrition.csv", features, target)
    print(w)    
    

itération 0 | erreur = 0.4906
itération 100 | erreur = 0.0526
itération 200 | erreur = 0.0526
itération 300 | erreur = 0.0526
itération 400 | erreur = 0.0526
itération 500 | erreur = 0.0526
itération 600 | erreur = 0.0526
itération 700 | erreur = 0.0525
itération 800 | erreur = 0.0525
itération 900 | erreur = 0.0525
itération 1000 | erreur = 0.0525
itération 1100 | erreur = 0.0525
itération 1200 | erreur = 0.0525
itération 1300 | erreur = 0.0525
itération 1400 | erreur = 0.0525
itération 1500 | erreur = 0.0524
itération 1600 | erreur = 0.0524
itération 1700 | erreur = 0.0524
itération 1800 | erreur = 0.0524
itération 1900 | erreur = 0.0524
itération 2000 | erreur = 0.0524
itération 2100 | erreur = 0.0524
itération 2200 | erreur = 0.0524
itération 2300 | erreur = 0.0523
itération 2400 | erreur = 0.0523
itération 2500 | erreur = 0.0523
itération 2600 | erreur = 0.0523
itération 2700 | erreur = 0.0523
itération 2800 | erreur = 0.0523
itération 2900 | erreur = 0.0523
[2.804701454474817e-07